<a href="https://colab.research.google.com/github/stephenebert/Springboard/blob/main/Mini_Project_Fine_tuning_a_Convolutional_Neural_Network_with_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Part 1: Preprocessing

**1. Load the CIFAR-10 dataset after referencing the documentation here.**

In [4]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

**2. Normalize the pixel values so they're all in the range [0, 1].**

In [5]:
print(int(max(x_train.max(), x_test.max())))
print(int(min(x_train.min(), x_test.min())))

255
0


In [6]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

In [44]:
print(int(max(x_train.max(), x_test.max())))
print(int(min(x_train.min(), x_test.min())))

1
0


We see that the pixel values are in the range [0, 1]

**3. Apply One Hot Encoding to the train and test labels using the to_categorical function.**

In [7]:
y_train = to_categorical(y_train,10)
y_test = to_categorical(y_test,10)
print(y_train.shape)
print(y_test.shape)

(50000, 10)
(10000, 10)


We see the one hot encoding works for the training and testing labels in CIFAR-10 when we print them. Note that we used Kera's to_categorical() method to convert integers 0, 1, ..., 9 to one-hot encoded vectors. num_classes = 10

In [46]:
print(f'y_train = \n {y_train}')
print('\n')
print(f'y_test = \n {y_test}')

y_train = 
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]]


y_test = 
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 1. 0. 0.]]


**4. Further split the the training data into training and validation sets using train_test_split. Use only 10% of the data for validation.**

For this, we just use the usual train_test_split from sklearn.model_selection.

In [8]:
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

# Part 2: VGG16 Setup

**1. Load VGG16 as a base model. Make sure to exclude the top layer.**

In [48]:
base_model = VGG16(weights = 'imagenet', include_top = False, input_shape = (32, 32, 3))

**2. Freeze all the layers in the base model. We'll be using these weights as a feature extraction layer to forward to layers that are trainable.**



In [49]:
for layer in base_model.layers:
    layer.trainable = False

The new top layers we'll add next will be the only trainable parts

# Part 3: Custom Classifier Layers

**1. Using the base model, add a GlobalAveragePooling2D layer, followed by a Dense layer of length 256 with ReLU activation. Finally, add a classification layer with 10 units, corresponding to the 10 CIFAR-10 classes, with softmax activation.**

In [50]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation = 'relu')(x)
predictions = Dense(10, activation = 'softmax')(x)

model = Model(inputs = base_model.input, outputs = predictions)

**2. Create a Keras Model that takes in approproate inputs and outputs.**

In [51]:
model = Model(inputs=base_model.input, outputs=predictions)

# Part 4: Compile and Train

**1. Compile your model using an appropriate loss function. Feel free to play around with the optimizer, but a good starting optimizer might be Adam with a learning rate of 0.001.**

In [52]:
model.compile(optimizer = Adam(learning_rate = 0.001), loss = 'categorical_crossentropy', metrics = ['accuracy'])

**2. Fit your model on the training data. Use the validation data to print the accuracy for each epoch. Try training for 10 epochs. Note, training can take a few hours so go ahead and grab a cup of coffee.**

In [53]:
history = model.fit(x_train, y_train, epochs=10, batch_size=32, validation_data=(x_val, y_val), verbose=1)

Epoch 1/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 407s 288ms/step - accuracy: 0.4694 - loss: 1.5039 - val_accuracy: 0.5794 - val_loss: 1.2220
Epoch 2/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 437s 285ms/step - accuracy: 0.5840 - loss: 1.1893 - val_accuracy: 0.5914 - val_loss: 1.1585
Epoch 3/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 444s 286ms/step - accuracy: 0.6072 - loss: 1.1261 - val_accuracy: 0.6034 - val_loss: 1.1364
Epoch 4/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 440s 285ms/step - accuracy: 0.6296 - loss: 1.0607 - val_accuracy: 0.6072 - val_loss: 1.1096
Epoch 5/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 403s 286ms/step - accuracy: 0.6463 - loss: 1.0055 - val_accuracy: 0.6104 - val_loss: 1.1167
Epoch 6/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 440s 285ms/step - accuracy: 0.6614 - loss: 0.9686 - val_accuracy: 0.6140 - val_loss: 1.1089
Epoch 7/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 443s 285ms/step - accuracy: 0.6721 - loss: 0.9384 - val_accuracy: 0.6182 - val_loss: 1.1065
Epoch 8/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 402s 285ms/step - ac

# Part 5: Evaluation and Experimentation

**1. Use your trained model to calculate the accuracy on the test set. Is the model performance better than random?**

In [54]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f'Test Accuracy: {test_accuracy}')

313/313 ━━━━━━━━━━━━━━━━━━━━ 83s 266ms/step - accuracy: 0.6101 - loss: 1.1205
Test Accuracy: 0.6132000088691711


Yes, this model's performance is better than random. Random guessing is about 10% so the fact our model is at around 61.32% is 6 times better than chance.

**2. Experiment! See if you can tweak your model to improve performance.**

In [13]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.optimizers import Adam

# Load MobileNetV2 with smaller input size
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96, 96, 3))

for layer in base_model.layers:
    layer.trainable = False

# custom top layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)



9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [15]:
TARGET_SIZE = 96
BATCH_SIZE = 64

def resize_and_batch(image, label):
    image = tf.image.resize(image, [TARGET_SIZE, TARGET_SIZE])
    return image, label

# Create resized and batched datasets
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val))

train_dataset = (
    train_dataset
    .map(resize_and_batch)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset
    .map(resize_and_batch)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [17]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model(tf.zeros((1, 96, 96, 3)))

history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset
)


Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 222s 308ms/step - accuracy: 0.6937 - loss: 0.8935 - val_accuracy: 0.7848 - val_loss: 0.6202
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 275s 327ms/step - accuracy: 0.8074 - loss: 0.5542 - val_accuracy: 0.7940 - val_loss: 0.5746
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 243s 301ms/step - accuracy: 0.8356 - loss: 0.4776 - val_accuracy: 0.7978 - val_loss: 0.5744
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 243s 345ms/step - accuracy: 0.8560 - loss: 0.4157 - val_accuracy: 0.7978 - val_loss: 0.5809
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 257s 339ms/step - accuracy: 0.8748 - loss: 0.3621 - val_accuracy: 0.7974 - val_loss: 0.6101
Epoch 6/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 265s 343ms/step - accuracy: 0.8934 - loss: 0.3119 - val_accuracy: 0.7948 - val_loss: 0.6348
Epoch 7/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 246s 321ms/step - accuracy: 0.9095 - loss: 0.2707 - val_accuracy: 0.7924 - val_loss: 0.6874
Epoch 8/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 277s 342ms/step - accuracy: 0.9211 -

We see that we get a higher accuracy at 93.47% with MobileNetV2 than the previous model using VGG16. Its architecture depthwise separable convolutions and skip connections is optimized for efficiency without losing accuracy.

Skip connections mean that we allow the model to pass info forward more easily, help gradients flow during training, and prevent the vanishing gradients in deep networks.

MobileNetV2 uses a special type of convolution that is much faster and lighter than regular convolutions. A regular convolution mixes spatial and channel information all at once whereas a depthwise separable convolution breaks it into 2 steps:

*   Depthwise convolution: applies one filter per channel

*   Pointwise convolution: combines all channels with 1 x 1 convolutions

This reduces the number of computations by a lot with little to no loss in performance.